# Преобразования и конвейеры

Авторы BelOAI. CC BY 4.0.

Выполняйте ячейки сверху вниз. Сначала решите задачи, затем обратитесь к решениям.

## Чему научимся

Собирать подготовку и модель в конвейер, обучающийся без доступа к проверочным данным.

## Данные и разбиение

В предыдущем уроке KNN использовал только размах. Теперь добавим массу с пропусками и категорию бумаги: подготовку каждого вида данных нужно обучить вместе с моделью внутри разбиения. Восстановим тот же учебный набор из 24 объектов и тот же тест. Поэтому новые результаты сравниваются на тех же объектах; это отдельный набор, а не CSV первого эксперимента. Классы по-прежнему обозначены числами 0 и 1.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.dummy import DummyClassifier

X = pd.DataFrame({
    "span_cm": list(range(10, 34)),
    "mass_g": [8., 9., 10., 11., None, 12.] * 4,
    "paper": ["тонкая", "плотная"] * 12,
})
y = np.array([0] * 12 + [1] * 12)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=17,
)
assert len(X_train) == 18 and len(X_test) == 6


## Разобранный пример: Pipeline и ColumnTransformer

Для одного числового столбца подготовка понятна: заполнить пропуск, затем привести значения к сравнимому масштабу. Но текст бумаги нельзя вычитать из среднего. Поэтому сначала разделим столбцы по виду, подготовим каждую группу своим способом и соединим результаты в одну числовую таблицу для KNN.

### Шаг 1. Числа и категории

`Pipeline` выполняет именованные шаги последовательно. `ColumnTransformer` применяет разные преобразования к разным столбцам. `SimpleImputer(strategy="median")` заполнит числовые пропуски медианой: центральным значением отсортированного ряда, а при чётной длине — средним двух центральных. Затем `StandardScaler` вычтет обучающее среднее и разделит на обучающее стандартное отклонение. Для чисел $x_1,\ldots,x_n$ со средним $m$ это $s=\sqrt{\frac1n\sum_i(x_i-m)^2}$: корень из среднего квадрата отклонений. Например, у `[1, 3]` среднее 2 и $s=1$, поэтому получим `[-1, 1]`. Постоянный обучающий столбец становится нулевым; вместо нулевого делителя библиотека использует 1. Для нового значения такого признака всё равно вычитается обучающее среднее.

В [масштабировании по диапазону](../../../math/linear-algebra/feature-scale/) мы вычитали минимум и делили на разность максимума и минимума. StandardScaler использует другое правило: среднее и стандартное отклонение. Оба способа меняют геометрию, и оба вычисляют свои параметры только по train; численные результаты у них не обязаны совпадать.

Категории бумаги превратим в отдельные числовые индикаторы через one-hot кодирование: например, «тонкая» и «плотная» станут двумя столбцами, а строка получит 1 в столбце своей категории и 0 в другом.

`OneHotEncoder` выполняет это кодирование. `sparse_output=False` просит обычный плотный массив: в нём хранятся все ячейки, включая нули. `handle_unknown="ignore"` даёт нулевые индикаторы для неизвестной категории, позволяя применить модель к новым данным. Это техническое правило, а не доказательство качества на новых категориях.

### Шаг 2. Проверка кандидатов

`GridSearchCV` сравнивает кандидатов на трёх внутренних разбиениях. В каждом разбиении `Pipeline` заново обучает заполнение, масштабирование и KNN только на внутренней обучающей части. Двойное подчёркивание в `knn__n_neighbors` указывает параметр шага `knn`. После выбора лучший конвейер автоматически обучается на всей `X_train`.

`KNeighborsClassifier` создаёт классификатор KNN, его параметр `n_neighbors` задаёт число соседей. `StratifiedKFold(n_splits=3, ...)` задаёт три группы с близкими долями классов: на каждом шаге из 18 обучающих объектов 12 используются для обучения и 6 для внутренней проверки. `shuffle=True` предварительно перемешивает строки.

`GridSearchCV` перебирает значения 1, 3 и 5; `scoring="accuracy"` выбирает долю верных ответов. `best_params_` после обучения хранит выбранные параметры. Функция `accuracy_score(правильные_ответы, предсказания)` вычисляет эту долю. Для сравнения `DummyClassifier(strategy="most_frequent")` запоминает наиболее частый обучающий класс и выдаёт его каждому объекту — это постоянный baseline.

### Шаг 3. Сборка и запуск

Список шагов `Pipeline` состоит из пар `(имя_шага, объект_шага)`. Для `ColumnTransformer` каждый элемент — тройка `(имя, преобразователь, список_столбцов)`. Имена задаём сами и затем используем для настройки параметров. При `fit` конвейер учит преобразователи по порядку и передаёт подготовленные данные модели; при `predict` применяет уже выученные преобразования.

Сначала создаём объекты шагов, затем объединяем их. Только вызов `search.fit` запускает обучение.

In [ ]:
numeric = Pipeline([
    ("fill", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
prepare = ColumnTransformer([
    ("numbers", numeric, ["span_cm", "mass_g"]),
    ("categories", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["paper"]),
])
model = Pipeline([
    ("prepare", prepare),
    ("knn", KNeighborsClassifier()),
])
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=17)
search = GridSearchCV(model, {"knn__n_neighbors": [1, 3, 5]}, cv=cv, scoring="accuracy")
search.fit(X_train, y_train)
predicted = search.predict(X_test)
assert predicted.shape == y_test.shape
print(search.best_params_)
print("test accuracy:", accuracy_score(y_test, predicted))
baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
assert accuracy_score(y_test, baseline.predict(X_test)) == 0.5


Ожидаем параметр из `{1, 3, 5}`, шесть предсказаний и baseline 0.5. Точный результат модели напечатает ноутбук: шесть тестовых объектов дают очень грубую оценку. Если после просмотра теста менять параметры, тест становится частью разработки и потребуется новая независимая проверка.

## Как обнаружить утечку

Опасный порядок: заполнить пропуски и масштабировать всю таблицу, затем разделить её. Даже без ответов преобразование уже использовало тест. Безопасный порядок: разделить исходные строки, затем передать обучение конвейеру. Сам конвейер не защищает от признака, вычисленного по правильным ответам, или одинаковых объектов по обе стороны разбиения.

Для справки: [типичные ошибки scikit-learn](https://scikit-learn.org/stable/common_pitfalls.html), [правило масштабирования StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html).

## Задачи

### 1. Базовая

Для чего нужен ColumnTransformer?

<details>
<summary>Подсказка</summary>

Разные столбцы требуют разных преобразований.

</details>



### 2. Базовая

Что означает knn__n_neighbors?

<details>
<summary>Подсказка</summary>

Первая часть — имя шага.

</details>



### 3. Прикладная

Проверьте форму и метки результата search.predict на X_test.

<details>
<summary>Подсказка</summary>

Каждый объект должен получить один ответ.

</details>



### 4. Прикладная

Подайте объект с новой бумагой и пропуском массы.

<details>
<summary>Подсказка</summary>

Имена столбцов должны совпадать.

</details>



### 5. Олимпиадная

Почему медиану нужно учить внутри каждого внутреннего разбиения?

<details>
<summary>Подсказка</summary>

Внутренняя проверка тоже должна быть невидимой подготовке.

</details>



### 6. Олимпиадная

Защищает ли Pipeline от признака, содержащего правильный ответ?

<details>
<summary>Подсказка</summary>

Он управляет порядком преобразований, а не смыслом признаков.

</details>



## Итоги

Вы умеете собирать подготовку разных столбцов и модель в конвейер для внутренней проверки.

## Самопроверка

Проследите пропущенную массу и новую категорию бумаги через все шаги. Какие строки доступны заполнению пропусков внутри одного разбиения? Когда разрешено открыть итоговый тест?

Для перехода дальше достаточно уверенно решать базовые и прикладные задачи. К олимпиадным можно вернуться позже.


## Решения задач

### Задача 1

Он применяет указанные преобразования к выбранным группам столбцов.

### Задача 2

Параметр числа соседей шага knn.

### Задача 3

In [ ]:
assert len(search.predict(X_test)) == len(X_test)
assert set(search.predict(X_test)).issubset({0, 1})


### Задача 4

In [ ]:
new_object = pd.DataFrame({"span_cm": [18], "mass_g": [None], "paper": ["новая"]})
assert search.predict(new_object).shape == (1,)


### Задача 5

Иначе значения проверочных строк повлияют на заполнение, а оценка кандидатов перестанет быть независимой.

### Задача 6

Нет. Нужно проверить происхождение каждого признака и его доступность при реальном предсказании.